<a href="https://colab.research.google.com/github/tmdoi/ai-measurement/blob/main/FFT_ver01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**0. 準備**

1.   microbitから取得したcsvファイルのファイル名を"data.csv"に変更する
2.   左側のファイルをクリックする
3.   ↑をクリックして data.csvをアップロードする




**1. データの読み込み**

　エラーが出る場合は，準備が完了しているか，再確認すること！

In [ ]:
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt

df = pd.read_csv("data.csv", sep="\t", skiprows=2)

print(df.head())

t =  df.iloc[:, 0].values
ax =  df.iloc[:, 1].values

**2. サンプリング周期の推定**

　マイクロビットで取得したデータの間隔は一定ではない．そのため，取得時間間隔を求めてその中から中央値を求める．

In [ ]:
dt = np.diff(t)
dt0 = np.median(dt)      # 中央値が安全
fs = 1.0 / dt0
print(dt0)
print(fs)

**3. 等間隔時刻列の生成**

　2. で取得した取得時間間隔の中央値を使って時間軸を生成する

In [ ]:
t_uniform = np.arange(t[0], t[-1], dt0)
print(t_uniform)

**4. 補間**

　マイクロビットで取得したデータの取得時間を一定の間隔で取得したデータとなるように線形補間している

In [ ]:
f_interp = interpolate.interp1d(t, ax, kind="linear")
ax_uniform = f_interp(t_uniform)
print(ax_uniform)

**4.1 補間結果の可視化**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

# 元の不等間隔データ（点）
plt.plot(t, ax, 'o', label="Original (non-uniform)", markersize=4)

# 補間後の等間隔データ（線）
plt.plot(t_uniform, ax_uniform, '-', label="Interpolated (uniform)")

plt.xlabel("Time [s]")
plt.ylabel("Acceleration (ax)")
plt.title("Comparison of Original and Interpolated Signals")
plt.legend()
plt.grid()

#plt.xlim(7, 9)  # 任意の時間区間

plt.tight_layout()
plt.show()


5. FFT

取得した時系列データの中から周期性をみつけて（FFT），その量を可視化する

In [ ]:
ax_uniform -= np.mean(ax_uniform)

N = len(ax_uniform)
fft_data = np.fft.fft(ax_uniform)
freq = np.fft.fftfreq(N, d=1/fs)

half = N // 2
plt.plot(freq[:half], np.abs(fft_data[:half]) / N)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")

#plt.xlim(0.1, 5) # 任意の時間区間

plt.grid()
plt.show()


**6. FFT結果から帯状ヒートマップ用データを作る**

**6.1 片側振幅スペクトルを取得**

In [ ]:
amp = np.abs(fft_data[:half]) / N
freq_pos = freq[:half]

**6.2  帯状に拡張（横方向に複製)**

In [ ]:
band_width = 10   # 帯の太さ（ピクセル）

heatmap_data = np.tile(
    amp[:, np.newaxis],
    (1, band_width)
)

** 6.3 帯状ヒートマップの描画 **

In [ ]:
plt.figure(figsize=(2, 6))

plt.imshow(
    heatmap_data,
    aspect="auto",
    origin="lower",
    extent=[0, band_width, freq_pos[0], freq_pos[-1]]
)

plt.colorbar(label="Amplitude")
plt.ylabel("Frequency [Hz]")
plt.xlabel("Dummy axis")
plt.title("Single FFT Band Heatmap")

plt.tight_layout()
plt.show()
